## Hospital Performance And Efficiancy Analysis
#### This project analyze hospital Performance using multiple healthcare dataset. The goal is to clean, prepare and analyze hospital data to understand the relationship between hospital ratings, medicare spending, and patient outcomes.

In [ ]:
# Importing necessary libraries to load, process, analyze and visualize the dataset.
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

### Hospital Information Data
##### The hospital_data contains the general information about hospitals. This dataset is foundational table and will later be joined with other datasets.

In [ ]:
# Loading hospital general information dataset
hospital_info = pd.read_csv("../data/Hospital_General_Information.csv")
hospital_info.head(5)

## EDA (Exploratory Data Analysis)
In this step The structure and contents of the dataset are examined to understand :
* Data types
* Check columns
* Missing values
* Basic statistics

In [ ]:
# checking datatypes 
hospital_info.info()

In [ ]:
# checking number of rows and columns
hospital_info.shape

In [ ]:
# Inspecting Columns
hospital_info.columns

In [ ]:
# checking null values
hospital_info.isnull().sum()

### Data Cleaning and Preparing: Hospital Info Data
In this section, hospital general information dataset is cleaned to prepare it for analysis. the cleaning process includes :
* Removing unnecessary columns
* Renaming columns 
* Handeling dublicates
* Replacing "Not Available" values with "NaN"
##### The goal is to create a clean dataset that can be easily used for analysis and combined with other datasets.


#### Removing Unnecessary Columns

In [ ]:
# Removing the unnecessary columns from hospital data to make dataset cleaner and easier to work during analysis.
hospital_info = hospital_info.drop(columns=['Address', 'City/Town',
       'ZIP Code', 'County/Parish', 'Telephone Number',
       'Meets criteria for birthing friendly designation',
        'Hospital overall rating footnote',
       'MORT Group Measure Count', 'Count of Facility MORT Measures',
       'Count of MORT Measures Better', 'Count of MORT Measures No Different',
       'Count of MORT Measures Worse', 'MORT Group Footnote',
       'Safety Group Measure Count', 'Count of Facility Safety Measures',
       'Count of Safety Measures Better',
       'Count of Safety Measures No Different',
       'Count of Safety Measures Worse', 'Safety Group Footnote',
       'READM Group Measure Count', 'Count of Facility READM Measures',
       'Count of READM Measures Better',
       'Count of READM Measures No Different', 'Count of READM Measures Worse',
       'READM Group Footnote', 'Pt Exp Group Measure Count',
       'Count of Facility Pt Exp Measures', 'Pt Exp Group Footnote',
       'TE Group Measure Count', 'Count of Facility TE Measures',
       'TE Group Footnote'], axis=1)
hospital_info.head(5)

#### Renaing Columns
##### In this step, A function was created to rename columns.The original dataset contains spaces and special character, which can cause error during analysis. columns are renamed using snake_case formatting.

In [ ]:
# column naming function
def rename_columns(df, new_names):
    return df.rename(columns=new_names)

In [ ]:
hospital_info= rename_columns(hospital_info,{
    'Facility ID' : 'facility_id',
    'Facility Name' : 'facility_name',
    'State': 'state',
    'Hospital Type' : 'hospital_type',
    'Hospital Ownership' : 'hospital_ownership',
    'Emergency Services' : 'emergency_services',
    'Hospital overall rating' : 'hospital_overall_rating'
    
})

#### Checking for Duplicates
In this step, dataset is checked for duplicate rows. Duplicate rows can cause error and may lead to double counting hospital data. Duplicate rows will be identified and removed.

In [ ]:
#check duplicates based on facility_id
duplicates_id = hospital_info.duplicated(subset=['facility_id']).sum()
duplicates_id 

Each hospital is uniquely represented in the dataset.

#### Replacing 'Not Available' to Nan

In [ ]:
# some rows in hospital_overall_rating column are 'Not Available'. I'm replacing that to NaN values and chenging datatype object to numeric.
hospital_info['hospital_overall_rating'] = pd.to_numeric(hospital_info['hospital_overall_rating'].replace('Not Available', np.nan))



In [ ]:
# List of unique states in hospital dataset
hospital_info['state'].unique()

##### Hospital_info dataset includes 56 unique states, which include 50 US states plus district of columnia(DC), and US territories : PR(PPuerto Rico), VI (Virgin Islands), AS (American Samoa), GU (Guam), MP (Northern Mariana Islands)

### Medicare Spending Data 
In this step, the **Medicare Spending** dataset is loaded and prepared for analysis. This dataset contains information about the Medicare spending for each hospital.

In [ ]:
# Loading medicare hospital spending per patient dataset
medicare_data = pd.read_csv("../data/Medicare_Hospital_spending_Per_Patient-Hospital.csv")
medicare_data.head(10)

In [ ]:
# checking datatypes and missing values
medicare_data.info()

In [ ]:
# total rows and columns
medicare_data.shape

### Data Cleaning: Medicare Info Data
In this section, medicare spending information dataset is cleaned to prepare it for analysis. the cleaning process includes :
* Reviewing columns names
* Inspecting null values
* Removing unnecessary columns
* Renaming columns 
* Checking for missing values
* Converting data types where necessary

In [ ]:
# columns in medicare dataset
medicare_data.columns

In [ ]:
# finding null values
medicare_data.isnull().sum()

##### Removing unnecessery columns

In [ ]:
# Removing unnecessery columns from medicare data
medicare_data = medicare_data.drop(columns=['Address', 'City/Town', 
       'ZIP Code', 'Facility Name', 'State','County/Parish','Measure ID', 'Telephone Number', 'Footnote'])
medicare_data.head(10)

In [ ]:
# some rows in Score column are 'Not Available". I'm replacing this to NaN, this allows numeric analysis.
medicare_data['Score']= medicare_data['Score'].replace('Not Available', np.nan)

# checking for the number of hospitals didn't report spending score
medicare_data['Score'].isna().sum()

##### out of 4627 hospitals in the dataset, 1728 do not have reported spending score, These missing values replaced with NaN to allow proper numeric analysis.

##### Renaming columns

In [ ]:
#renaming using previously created function
medicare_data = rename_columns(medicare_data, {
   'Facility ID' : 'facility_id',
    'Measure Name' : 'measure_name',
    'Score' : 'spending_score',
    'Start Date' : 'start_date',
    'End Date' : 'end_date'
})

##### Inspecting duplicates

In [ ]:
# checking duplicate rows
medicare_data.duplicated(subset=['facility_id']).sum()

In [ ]:
# checking the data types
medicare_data.dtypes

##### Data type conversion and custom Function:
* Numeric datatype to object
* Function to convert object to numeric
* Function to convert object to datetime


In [ ]:
# converting the facility_id to string as IDs are identifiers not numbers.
medicare_data['facility_id'] = medicare_data['facility_id'].astype(str)


In [ ]:
# Function to convert object to numeric datatype
def convert_to_numeric(df, columns):
    for column in columns:
        df[column]=pd.to_numeric(df[column])
    return df

# converting spending score to numeric datatype.
medicare_data = convert_to_numeric(medicare_data, ['spending_score'])

In [ ]:
# Function to convert string date to datetime datatype.
def convert_to_datetime(df,columns):
    for column in columns:
        df[column] = pd.to_datetime(df[column])
    return df

#converting start_date and end_date to datetime
medicare_data= convert_to_datetime(medicare_data,['start_date', 'end_date'])

In [795]:
# overview of medicare_data after cleaning dataset
medicare_data['spending_score'].describe()


count    2899.000000
mean        0.991852
std         0.084037
min         0.510000
25%         0.940000
50%         0.990000
75%         1.040000
max         1.510000
Name: spending_score, dtype: float64

### Unplanned Hospital Visits
In this step **Unplanned Hospital Visits dataset** is loaded and prepared for analysis. This dataset contains information about patients and also the patients who returned for unplanned visits.


In [ ]:
# Loading unplanned hospital visits
unplanned_visit_data= pd.read_csv("../data/Unplanned_Hospital_visits-Hospital.csv")
unplanned_visit_data.head(5)

# The unplanned visit dataset contains the information about hospital visits or return visits for different medical conditions.

#### Data Cleaning and Preparation: unplanned_visit_data
The cleaning process includes:
* Checking the structure and contents of dataset
* Checking for missing values
* Removing unnecessery columns
* Converting data types
* Renaming columns for better understanding
* Handling duplicates



In [799]:
unplanned_visit_data.dtypes

facility_id                     object
measure_name                    object
compared_to_national            object
visit_score                    float64
num_patients                   float64
num_patient_returned           float64
start_date              datetime64[ns]
end_date                datetime64[ns]
dtype: object

In [ ]:
# Unplanned  visit rows and columns
unplanned_visit_data.shape

In [ ]:
#columns in unplanned visit data
unplanned_visit_data.columns

In [ ]:
# checking for null values
unplanned_visit_data.isnull().sum()

In [ ]:
# removing unnecessery columns from unplanned visit data
unplanned_visit_data = unplanned_visit_data.drop(columns=['Address', 'City/Town',
       'ZIP Code', 'County/Parish', 'Telephone Number', 'Measure ID',
         'Denominator', 'Facility Name',	'State',
       'Lower Estimate', 'Higher Estimate', 
        'Footnote'])
unplanned_visit_data.head(5)

In [ ]:
# some rows in Number of Patients and Number of Patients Returned_returned columns have 'Not Applicable. I'm replacing this to NaN, this allows numeric analysis.
unplanned_visit_data['Number of Patients'] = unplanned_visit_data['Number of Patients'].replace('Not Applicable', np.nan)
unplanned_visit_data['Number of Patients Returned']= unplanned_visit_data['Number of Patients Returned'].replace('Not Applicable', np.nan)

# some rows in Score, number of patients and number of patients columns have 'Not Available'. replacing this to NaN.
unplanned_visit_data[['Score', 'Number of Patients', 'Number of Patients Returned']] = unplanned_visit_data[['Score', 'Number of Patients', 'Number of Patients Returned']].replace('Not Available', np.nan)

In [ ]:
# renaming columns cleanly
unplanned_visit_data = rename_columns(unplanned_visit_data, {
    'Facility ID' : 'facility_id',
    'Measure Name' : 'measure_name',
    'Compared to National' : 'compared_to_national',
    'Score' : 'visit_score',
    'Number of Patients': 'num_patients',
    'Number of Patients Returned': 'num_patient_returned',
    'Start Date': 'start_date',
    'End Date' : 'end_date'
})
unplanned_visit_data.head(5)


In [ ]:
# checking duplicate rows
unplanned_visit_data.duplicated().sum()

In [ ]:
#converting start_date and end_date to datetime calling function

unplanned_visit_data= convert_to_datetime(unplanned_visit_data,['start_date', 'end_date'])

#converting visit_score, num_patients, and num_patients returns from string to numeric data type calling function
unplanned_visit_data = convert_to_numeric(unplanned_visit_data,['visit_score','num_patients','num_patient_returned'])

In [796]:
# Summary statics for unplanned visit data
unplanned_visit_data.describe()

,visit_score,num_patients,num_patient_returned,start_date,end_date
count,35358.000000,8352.000000,8352.000000,67074,67074
mean,11.147412,225.679478,65.376437,2021-12-04 20:34:17.142857216,2024-05-09 00:00:00
min,-90.700000,17.000000,1.000000,2021-01-01 00:00:00,2023-12-31 00:00:00
25%,5.200000,69.000000,20.000000,2021-07-01 00:00:00,2023-12-31 00:00:00
50%,13.800000,151.000000,43.000000,2021-07-01 00:00:00,2024-06-30 00:00:00
75%,17.000000,305.000000,88.000000,2023-01-01 00:00:00,2024-06-30 00:00:00
max,247.400000,3216.000000,959.000000,2023-07-01 00:00:00,2024-06-30 00:00:00
std,15.735873,230.499300,67.767445,NaN,NaN


In [797]:
# unique measure name in unplanned visit dataset
unplanned_visit_data['measure_name'].unique()

array(['Hospital return days for heart attack patients',
       'Hospital return days for heart failure patients',
       'Hospital return days for pneumonia patients',
       'Hybrid Hospital-Wide All-Cause Readmission Measure (HWR)',
       'Rate of unplanned hospital visits after colonoscopy (per 1,000 colonoscopies)',
       'Rate of inpatient admissions for patients receiving outpatient chemotherapy',
       'Rate of emergency department (ED) visits for patients receiving outpatient chemotherapy',
       'Ratio of unplanned hospital visits after hospital outpatient surgery',
       'Acute Myocardial Infarction (AMI) 30-Day Readmission Rate',
       'Rate of readmission for CABG',
       'Rate of readmission for chronic obstructive pulmonary disease (COPD) patients',
       'Heart failure (HF) 30-Day Readmission Rate',
       'Rate of readmission after hip/knee replacement',
       'Pneumonia (PN) 30-Day Readmission Rate'], dtype=object)

In [800]:
# Average visit score grouped by measure name
unplanned_visit_data.groupby('measure_name')['visit_score'].mean().sort_values(ascending=False).round(2)

measure_name
Heart failure (HF) 30-Day Readmission Rate                                                 19.71
Rate of readmission for chronic obstructive pulmonary disease (COPD) patients              18.23
Pneumonia (PN) 30-Day Readmission Rate                                                     15.99
Hybrid Hospital-Wide All-Cause Readmission Measure (HWR)                                   14.98
Acute Myocardial Infarction (AMI) 30-Day Readmission Rate                                  13.63
Rate of unplanned hospital visits after colonoscopy (per 1,000 colonoscopies)              13.07
Rate of inpatient admissions for patients receiving outpatient chemotherapy                10.78
Rate of readmission for CABG                                                               10.64
Hospital return days for heart attack patients                                              7.14
Hospital return days for pneumonia patients                                                 6.22
Rate of emergency

The analysis above calculated the average visit score for each measure category. It helps to identify which types of measure have higher or lower visit scores. The measures are sorted descending order to highlight the measures with the highest average score.

### Connect to SQLite Database
In this step connection to a **SQL Database** is established. This allows cleaned dataset to be stored in a structured format for efficient querying and analysis

In [ ]:
# Creating connection
connection = sqlite3.connect("../data/hospital_performance.db")
cursor = connection.cursor()

# Loading Cleaned datasets into SQL database as seperate tables.
hospital_info.to_sql('hospital_info', connection, if_exists='replace', index=False)
medicare_data.to_sql('medicare_spending', connection, if_exists='replace', index=False)
unplanned_visit_data.to_sql('unplanned_visits', connection, if_exists='replace', index=False)


In [ ]:
# varifying the tables to confirm the database is successfully created.
pd.read_sql("SELECT * FROM hospital_info", connection)
pd.read_sql("SELECT * FROM medicare_spending", connection)
pd.read_sql("SELECT * FROM unplanned_visits ", connection)



### SQL quaries
##### I'm using sql quaries to explore relationships between hospital charasteristic, Medicare spending and patients visits. The goal of this analysis is to investigate patterns in hospital performance and efficency.
##### The following questions guide the analysis:
* Do hospital with higher rating spend more or less on Medicare service?
* Which States have the highest rating hospitals?
* Top 10 Hospitals with high medicare spending and high rating
* Which hospital have the higher rates of unplanned patient return?
* Hospital Efficiency Classification Based on Spending and Patient Outcomes
* Do hospitals that provide Imergency services have different performace rating?



#### Query: 1. Do hospitals with higher rating spend more or less on Medicare service?

In [ ]:
query1 = ''' select
    h.hospital_overall_rating,
    avg(m.spending_score) as avg_spending
from hospital_info h
join medicare_spending m
    on h.facility_id = m.facility_id
group by h.hospital_overall_rating
order by h.hospital_overall_rating;
'''
result1 = pd.read_sql(query1, connection)
result1

In [ ]:
# bar chart to visulize relation between hospital overall rating and average spending

# remove hospital with no rating for better trend
clean_result= result1[result1['hospital_overall_rating'] != 'Not Available'] 


plt.bar(
    clean_result['hospital_overall_rating'], 
    clean_result['avg_spending'],
    color='green',
    alpha=0.6
    )
plt.ylim(0.95, 1.05) # adjusted the y-axis to better highlight small variation on spending scores.
 
plt.xlabel("Hospital Overall Rating", fontsize=12, fontweight='bold')
plt.ylabel('Medicare Average Spending', fontsize=12, fontweight='bold')
plt.title("Average Medicare Spending vs Hospital Rating", fontsize=16, fontweight='bold', pad=30)

# to remove the top and right spine
ax=plt.gca()
ax.spines['top'].set_visible(False) 
ax.spines['right'].set_visible(False) 


plt.show()

##### This bar chart compares the average Medicare spending across diffrent hospital rating level. Each bar represents a rating category help us to observe how spending varies with hospital performance. The chart shows that higher rated hospitals have lower average Medicare spending compared to lower rated hospitals. This suggests the hospitals with better ratings may deliver more efficient care, potentially reducing the need of additional treatments and lowering overall healthcare costs.

#### Query2: Which States have the highest rating hospitals?


In [ ]:
query2 = '''select
    state,
    hospital_overall_rating
from hospital_info
group by state
having hospital_overall_rating > 3
order by hospital_overall_rating desc
;   
'''
result2 = pd.read_sql(query2, connection)
result2

#### Query3: Top 10 Hospitals with high medicare spending and high rating

In [ ]:
query3 = '''select 
    h.facility_name,
    h.hospital_overall_rating,
    m.spending_score
from hospital_info h
join medicare_spending m
    on h.facility_id = m.facility_id
where m.spending_score > (
        select avg(spending_score)
        from medicare_spending)
and h.hospital_overall_rating >= 4
order by m.spending_score desc
limit 10
;
'''
result3 = pd.read_sql(query3, connection)
result3

#### Query4: Which hospital have the higher rates of unplanned patient return?

In [ ]:
query4 = '''select
    h.facility_id,
    h.facility_name,
    sum(u.num_patient_returned) as num_patient_returned
from hospital_info h
join unplanned_visits u
    on h.facility_id = u.facility_id
group by h.facility_id, facility_name
order by num_patient_returned desc
limit 10;

'''
result4 = pd.read_sql(query4, connection)
result4


The analysis shows top 10 hospitals having highest patients returned, which highlights potential issues in patients care and follow-up processes.

#### Query5: Wheather high spending hospitals have higher unplanned visits?

In [ ]:
query5 = '''select
    h.facility_id,
    h.facility_name,
    avg(m.spending_score) as avg_spending,
    round(sum(u.num_patient_returned)/sum(u.num_patients),3)*100 as return_rate
from hospital_info h
join medicare_spending m
    on h.facility_id = m.facility_id
join unplanned_visits u
    on h.facility_id = u.facility_id
WHERE u.end_date >= '2023-01-01'
AND u.start_date <= '2023-12-31'
group by h.facility_id, h.facility_name
HAVING SUM(u.num_patients) > 0
order by avg_spending desc
limit 20;

'''
result5 = pd.read_sql(query5, connection)
result5

##### The analysis above shows that hospital with higher medicare spending do not always achieve lower patient return rates. Some high spending hospitals have high return rates. This suggests that increased spending alone doesn't gurantee improved patients outcomes.

In [ ]:
plt.Figure(figsize=(14,8))
plt.scatter(result5['avg_spending'], result5['return_rate'], 
            color = 'red', 
            alpha=0.8,          # transparency of dots
            s=25,)               #size of dots

#labels
plt.xlabel("Average Spending",
           fontsize=12,
           fontweight= 'bold')
plt.ylabel("Return Rate (%)",
           fontsize=12,
           fontweight= 'bold')
plt.title("Does Higher Medicare Spending Reduce Patient Return Rates? (2023)",
          fontsize=16,
           fontweight='bold',
           pad= 20,
           y=0.98)  #position of title

ax=plt.gca()
ax.spines['top'].set_visible(False) 
ax.spines['right'].set_visible(False) 

plt.show()



##### The scatter plot above shows the relationship between average medicare spending and patient return rates of the top 20 hospitals with higher spending. Each dots represents a hospital comparing how much is spent with how frequently patients return.
##### There is no strong correlation between Medicare spending and patient return rates. Hospitals with similar spending level also have different return rates. This suggests that spending more doesn't always reduce patient outcome.

#### Query6: Hospital Efficiency Classification Based on Spending and Patient Outcomes

In [ ]:
query6 = '''
select
    h.facility_name,
    avg(m.spending_score) as avg_spending,
    round(sum(u.num_patient_returned)/sum(u.num_patients) * 100, 2)as return_rate,
    
    case
        when avg(m.spending_score) > (
                select avg(spending_score)
                from medicare_spending)
            and round(sum(u.num_patient_returned)/sum(u.num_patients) * 100, 2) > 35.00
        then "High Cost - Poor Outcome"
        
        when avg(m.spending_score) < (
                select avg(spending_score)
                from medicare_spending)
            and round(sum(u.num_patient_returned)/sum(u.num_patients) * 100, 2) < 35.00
        then "Efficient Hospital"
        
        else "Average Hospital"
        
    end as hospital_performance
from hospital_info h
join medicare_spending m
    on h.facility_id = m.facility_id
join unplanned_visits u
    on h.facility_id = u.facility_id
WHERE u.end_date >= '2023-01-01'
AND u.start_date <= '2023-12-31'
group by h.facility_name
HAVING SUM(u.num_patients) > 0 and avg(m.spending_score) IS NOT NULL
;
    
'''


result6 = pd.read_sql(query6, connection)
result6

In [ ]:
# to better understand the overall hospital distribution of hospital performance, I'm calculating the count and percentage of hospital in each category
 
performance_summary = result6['hospital_performance']. value_counts(). reset_index()   # counts the hospital in each categoru and converts into a table
performance_summary.columns = ['category', 'count']   # rename columns

performance_summary['percentage'] = round((performance_summary['count'] / performance_summary['count'].sum())*100,2)  #percentage
performance_summary

In [ ]:
# Piechart 
plt.figure(figsize=(6,6))
plt.pie(performance_summary['percentage'],
        labels = performance_summary['category'],
        startangle=90, 
         colors=['#66c2a5','#fc8d62','#8da0cb'],
        autopct="%1.2f%%")

plt.title('Distribution of Hospital Performance',
          fontsize=16,
          fontweight='bold',
          pad=20,
          y=0.98)

plt.legend(title="Hospital Performance",
           loc='lower right',
           bbox_to_anchor=(0.2, -0.1),  #moves below chart
           fontsize='small'
           
           )
plt.show()

The chart above shows the majority of hospitals (48.91%) fall into the Efficient and 47.43% Average category, while only small percentage(3.65%) are classified as high cost and poor outcome hospital. This indicates that most hospitals perform reasonably well, with very few demonstrating significantly poor performance.

#### Query7: Do better rated hospital actually have better patient outcomes?

In [793]:
query7 = ''' select 
    h.hospital_overall_rating,
    round(sum(u.num_patient_returned)/sum(u.num_patients) * 100, 2)as return_rate
from hospital_info h
join unplanned_visits u
    on h.facility_id = u.facility_id
where  h.hospital_overall_rating IS NOT NULL
group by h.hospital_overall_rating
order by h.hospital_overall_rating;
'''

result7 = pd.read_sql(query7, connection)
result7

,hospital_overall_rating,return_rate
0,1.0,30.65
1,2.0,29.59
2,3.0,29.17
3,4.0,28.48
4,5.0,27.93
